# Zero-shot 28種の調査

train_soundscapes_labels.csv と taxonomy.csv を分析し、
ゼロショット28種の情報を収集する。

**GPU不要、CPU + Internet OFF で実行可能**

In [ ]:
import os
import pandas as pd
import numpy as np

COMP = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP = p; break
assert COMP
print(f'Data: {COMP}')
print(f'Files: {os.listdir(COMP)}')

In [ ]:
# ============================================================
# 1. ゼロショット28種を特定
# ============================================================
train = pd.read_csv(f'{COMP}/train.csv')
sub = pd.read_csv(f'{COMP}/sample_submission.csv')

sub_species = set(sub.columns) - {'row_id'}
train_species = set(train['primary_label'].astype(str).unique())
zero_shot = sorted(sub_species - train_species)

print(f'提出要求種: {len(sub_species)}')
print(f'訓練データ種: {len(train_species)}')
print(f'ゼロショット種: {len(zero_shot)}')
print()
for sp in zero_shot:
    print(f'  {sp}')

In [ ]:
# ============================================================
# 2. train_soundscapes_labels.csv の分析
# ============================================================
ts_path = f'{COMP}/train_soundscapes_labels.csv'
if os.path.exists(ts_path):
    ts = pd.read_csv(ts_path)
    print(f'train_soundscapes_labels.csv')
    print(f'  Shape: {ts.shape}')
    print(f'  Columns: {ts.columns.tolist()}')
    print()
    print(ts.head(10))
else:
    print(f'NOT FOUND: {ts_path}')
    # 別名の可能性
    for f in os.listdir(COMP):
        if 'soundscape' in f.lower() or 'label' in f.lower():
            print(f'  Found: {f}')

In [ ]:
# ============================================================
# 3. train_soundscapes にゼロショット種が含まれるか？
# ============================================================
if os.path.exists(ts_path):
    # primary_label 列を探す
    label_col = None
    for col in ts.columns:
        if 'label' in col.lower() or 'species' in col.lower() or 'primary' in col.lower():
            label_col = col
            break
    
    if label_col is None:
        print('Label column not found. Columns:', ts.columns.tolist())
        # 全列の内容を確認
        for col in ts.columns:
            print(f'\n{col}: unique={ts[col].nunique()}, sample={ts[col].head(3).tolist()}')
    else:
        ts_species = set(ts[label_col].astype(str).unique())
        found = sorted(set(zero_shot) & ts_species)
        not_found = sorted(set(zero_shot) - ts_species)
        
        print(f'Label column: {label_col}')
        print(f'train_soundscapes の種数: {len(ts_species)}')
        print(f'\n=== ゼロショット種のうち train_soundscapes に含まれるもの: {len(found)} / {len(zero_shot)} ===')
        for sp in found:
            count = len(ts[ts[label_col].astype(str) == sp])
            print(f'  {sp}: {count} rows')
        
        print(f'\n=== train_soundscapes にも含まれないもの: {len(not_found)} ===')
        for sp in not_found:
            print(f'  {sp}')

In [ ]:
# ============================================================
# 4. taxonomy.csv からゼロショット28種の情報を取得
# ============================================================
tax_path = f'{COMP}/taxonomy.csv'
if os.path.exists(tax_path):
    tax = pd.read_csv(tax_path)
    print(f'taxonomy.csv')
    print(f'  Shape: {tax.shape}')
    print(f'  Columns: {tax.columns.tolist()}')
    print()
    
    print('=== ゼロショット28種の taxonomy 情報 ===')
    for sp in zero_shot:
        row = tax[tax['primary_label'].astype(str) == sp]
        if len(row) > 0:
            info = row.iloc[0]
            print(f'\n{sp}:')
            for col in tax.columns:
                print(f'  {col}: {info[col]}')
        else:
            print(f'\n{sp}: NOT FOUND in taxonomy.csv')
else:
    print(f'NOT FOUND: {tax_path}')

In [ ]:
# ============================================================
# 5. train_soundscapes ディレクトリの確認
# ============================================================
ts_dir = f'{COMP}/train_soundscapes'
if os.path.isdir(ts_dir):
    files = os.listdir(ts_dir)
    ogg_files = [f for f in files if f.endswith('.ogg')]
    print(f'train_soundscapes/')
    print(f'  Total files: {len(files)}')
    print(f'  OGG files: {len(ogg_files)}')
    print(f'  Sample: {ogg_files[:5]}')
else:
    print(f'NOT FOUND: {ts_dir}')

In [ ]:
# ============================================================
# 6. まとめ
# ============================================================
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'ゼロショット種: {len(zero_shot)}')
if os.path.exists(ts_path) and label_col:
    print(f'train_soundscapes でカバー: {len(found)} / {len(zero_shot)}')
    print(f'完全に未カバー: {len(not_found)}')
print()
print('Next steps:')
print('  - カバーされた種 → train_soundscapes から音声切り出し + Embedding 抽出')
print('  - 未カバーの種 → 外部データ or Perch logit 活用')